In [1]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from features.ema_spreads import EmaSpreadSpec, add_ema_spread_features
from features.cross_through import CrossThroughSpec, add_cross_through_features
from features.stochastic_signals import add_stochastic_signal_features
from features.ema_compression import EMACompressionSpec, add_ema_compression_features, range_slope_strong_for_n_tf_candles
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)

from ema_diagnostic_plots import (
    ema_pair_spread_panel,
    ema_group_spread_panel,
    ema_cut_through_panel,
)
from simulation.simulator import (
    TradeExitConfig,
    StopLossConfig,
    PositionSizingConfig,
    TakeProfitConfig,
)

In [2]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2025-11-01"
sim_start = "2026-05-06"
end = None

In [3]:
# ============================================================
# STOCHASTIC Configs
# ============================================================

STOCH_CFG = {
    "k_length": 14,
    "d_length": 3,
    "smooth": 3,
}

In [4]:
### 1 Minute Indicators ###
EMA20_1m = "1m__ema__EMA_20"
EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
EMA200_1m = "1m__ema__EMA_200"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"
STO_K_1M = "1m__st14__K"
STO_D_1M = "1m__st14__D"
STO_1M_UP20_ACTIVE_5 = "1m__st14__K_CROSS_UP_20_ACTIVE_5"
STO_1M_DOWN20_ACTIVE_5 = "1m__st14__K_CROSS_DOWN_20_ACTIVE_5"

### 5 Minute Indicators ###
EMA20_5m = "5m__ema__EMA_20"
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"
STO_K_5M = "5m__st14__K"
STO_D_5M = "5m__st14__D"
STO_5M_UP20_ACTIVE_3 = "5m__st14__K_CROSS_UP_20_ACTIVE_3"
STO_5M_DOWN20_ACTIVE_3 = "5m__st14__K_CROSS_DOWN_20_ACTIVE_3"


### 15 Minute Indicators ###
EMA20_15m = "15m__ema__EMA_20"
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"
STO_K_15M = "15m__st14__K"
STO_D_15M = "15m__st14__D"
STO_15M_UP20_ACTIVE_3 = "15m__st14__K_CROSS_UP_20_ACTIVE_3"
STO_15M_DOWN20_ACTIVE_3 = "15m__st14__K_CROSS_DOWN_20_ACTIVE_3"

### 1 Hour Indicators ###
EMA20_1h = "1h__ema__EMA_20"
EMA50_1h = "1h__ema__EMA_50"
EMA75_1h = "1h__ema__EMA_75"
EMA100_1h = "1h__ema__EMA_100"
EMA125_1h = "1h__ema__EMA_125"
EMA150_1h = "1h__ema__EMA_150"
EMA175_1h = "1h__ema__EMA_175"
EMA200_1h = "1h__ema__EMA_200"
MACD_1H = "1h__macd_8_21_5__MACD"
MACD_SIGNAL_1H = "1h__macd_8_21_5__SIGNAL"
MACD_HIST_1H = "1h__macd_8_21_5__HIST"
BULL_DIV_1h = "1h__rsi14__BULL_DIV"
BEAR_DIV_1h = "1h__rsi14__BEAR_DIV"
BULL_START_RSI_1h = "1h__rsi14__BULL_START_RSI"
BEAR_START_RSI_1h = "1h__rsi14__BEAR_START_RSI"
STO_K_1H = "1h__st14__K"
STO_D_1H = "1h__st14__D"
STO_1H_UP20_ACTIVE_3 = "1h__st14__K_CROSS_UP_20_ACTIVE_3"
STO_1H_DOWN20_ACTIVE_3 = "1h__st14__K_CROSS_DOWN_20_ACTIVE_3"

In [5]:
# ============================================================
# 1m indicators
# ============================================================

ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 100, 150, 200, 250],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]


# ============================================================
# 5m indicators
# ============================================================

ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]


# ============================================================
# 15m indicators
# ============================================================

ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]

# ============================================================
# 1h indicators
# ============================================================

ind_1h = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]

In [6]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

# 1h filter data
df60 = fetch_binance_klines(
    symbol=symbol,
    interval="1h",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df60_feat, _, _ = build_feature_df(
    raw_df=df60,
    tz=tz,
    ma_windows=[],
    indicators=ind_1h,
)


# Add Stochastic Features
df1_feat = add_stochastic_signal_features(
    df1_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2, 3, 5),
    source="K",
)

df5_feat = add_stochastic_signal_features(
    df5_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1,2, 3, 5),
    source="K",
)

df15_feat = add_stochastic_signal_features(
    df15_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2, 3, 5),
    source="K",
)

df60_feat = add_stochastic_signal_features(
    df60_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2),
    source="K",
)

merged_tmp = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
        "1h": df60_feat,
    },
    base_label="1m",
)


[06:23:12] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1m, start=2025-11-01 00:00:00+00:00, end=LATEST
[06:23:12] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance)
[06:23:12] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_1m.parquet
[06:23:12] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.partials\binance_spot_BTCUSDT_1m
[06:23:12] INFO | 05. Cache: HIT | rows=295,258 (main_rows=295,258, partial_files=0) | range=2025-11-01 00:00:00+00:00 -> 2026-05-25 00:57:00+00:00
[06:23:12] INFO | 06. Fetch plan 1/1: 2026-05-25 00:58:00+00:00 -> LATEST
[06:23:12] INFO | 07. Starting fetch range 1/1.


BTCUSDT 1m: 0page [00:00, ?page/s]

[06:23:13] INFO | 08. Completed fetch range 1/1 | rows=26 | total_rows_fetched=26 | requests=1 | retries=0 | elapsed=1.0s
[06:23:13] INFO | 09. Cache consolidated and saved (parquet) | rows=295,284 | range=2025-11-01 00:00:00+00:00 -> 2026-05-25 01:23:00+00:00 | cleared_partial_files=1
[06:23:13] INFO | 10. Fetch summary | new_rows=26 | successful_pages=1 | requests=1 | retries=0 | partial_saves=1 | elapsed=1.3s
[06:23:13] INFO | 11. Returning slice | rows=295,284 | range=2025-11-01 00:00:00+00:00 -> 2026-05-25 01:23:00+00:00
[06:23:13] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=5m, start=2025-11-01 00:00:00+00:00, end=LATEST
[06:23:13] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance)
[06:23:13] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_5m.parquet
[06:23:13] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data

BTCUSDT 5m: 0page [00:00, ?page/s]

[06:23:14] INFO | 08. Completed fetch range 1/1 | rows=5 | total_rows_fetched=5 | requests=1 | retries=0 | elapsed=0.8s
[06:23:15] INFO | 09. Cache consolidated and saved (parquet) | rows=146,609 | range=2025-01-01 00:00:00+00:00 -> 2026-05-25 01:20:00+00:00 | cleared_partial_files=1
[06:23:15] INFO | 10. Fetch summary | new_rows=5 | successful_pages=1 | requests=1 | retries=0 | partial_saves=1 | elapsed=1.1s
[06:23:15] INFO | 11. Returning slice | rows=59,057 | range=2025-11-01 00:00:00+00:00 -> 2026-05-25 01:20:00+00:00
[06:23:15] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=15m, start=2025-11-01 00:00:00+00:00, end=LATEST
[06:23:15] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance)
[06:23:15] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_15m.parquet
[06:23:15] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.

BTCUSDT 15m: 0page [00:00, ?page/s]

[06:23:15] INFO | 08. Completed fetch range 1/1 | rows=2 | total_rows_fetched=2 | requests=1 | retries=0 | elapsed=0.7s
[06:23:15] INFO | 09. Cache consolidated and saved (parquet) | rows=19,686 | range=2025-11-01 00:00:00+00:00 -> 2026-05-25 01:15:00+00:00 | cleared_partial_files=1
[06:23:15] INFO | 10. Fetch summary | new_rows=2 | successful_pages=1 | requests=1 | retries=0 | partial_saves=1 | elapsed=0.8s
[06:23:15] INFO | 11. Returning slice | rows=19,686 | range=2025-11-01 00:00:00+00:00 -> 2026-05-25 01:15:00+00:00
[06:23:15] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1h, start=2025-11-01 00:00:00+00:00, end=LATEST
[06:23:15] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance)
[06:23:15] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_1h.parquet
[06:23:15] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.par

In [8]:
# ============================================================
# EMA compression feature names
# ============================================================

EMA_1M_COMP_NAME = "ema_1m_50_100_150_comp"
EMA_5M_COMP_NAME = "ema_5m_50_100_150_comp"
EMA_15M_COMP_NAME = "ema_15m_50_100_150_comp"
EMA_1H_COMP_NAME = "ema_1h_50_100_150_comp"

EMA_1M_RANGE = f"{EMA_1M_COMP_NAME}__range_pct"
EMA_5M_RANGE = f"{EMA_5M_COMP_NAME}__range_pct"
EMA_15M_RANGE = f"{EMA_15M_COMP_NAME}__range_pct"

N_1M_SLOPE = 3
N_5M_SLOPE = 2
N_15M_SLOPE = 2

# ============================================================
# Add EMA compression features for 5m, 15m, 1h
# ============================================================

merged = add_ema_compression_features(
    merged_tmp,
    specs=[
        EMACompressionSpec(
            name=EMA_1M_COMP_NAME,
            cols=[EMA50_1m, EMA100_1m, EMA150_1m],
            compress_thr=0.35,
            expand_thr=0.40,
            lookback=20,
            require_bullish_order=False,
            fresh_only=False,
        ),
        EMACompressionSpec(
            name=EMA_5M_COMP_NAME,
            cols=[EMA50_5m, EMA100_5m, EMA150_5m],
            compress_thr=0.15, #0.15
            expand_thr=0.3, #0.3
            lookback=20, #20
            require_bullish_order=True,
            fresh_only=False,
        ),

        EMACompressionSpec(
            name=EMA_15M_COMP_NAME,
            cols=[EMA50_15m, EMA100_15m, EMA150_15m],
            compress_thr=0.10,
            expand_thr=0.15,
            lookback=20,
            require_bullish_order=True,
            fresh_only=False,
        ),

        EMACompressionSpec(
            name=EMA_1H_COMP_NAME,
            cols=[EMA50_1h, EMA100_1h, EMA150_1h],
            compress_thr=0.10,
            expand_thr=0.15,
            lookback=20,
            require_bullish_order=True,
            fresh_only=False,
        ),
    ],
)


C:\Users\Talal Zahid\Desktop\Talal\Binance\features\ema_compression.py:64: RuntimeWarning: All-NaN slice encountered
  row_max = np.nanmax(mat, axis=1)
C:\Users\Talal Zahid\Desktop\Talal\Binance\features\ema_compression.py:65: RuntimeWarning: All-NaN slice encountered
  row_min = np.nanmin(mat, axis=1)
C:\Users\Talal Zahid\Desktop\Talal\Binance\features\ema_compression.py:66: RuntimeWarning: Mean of empty slice
  row_mean = np.nanmean(mat, axis=1)
C:\Users\Talal Zahid\Desktop\Talal\Binance\features\ema_compression.py:64: RuntimeWarning: All-NaN slice encountered
  row_max = np.nanmax(mat, axis=1)
C:\Users\Talal Zahid\Desktop\Talal\Binance\features\ema_compression.py:65: RuntimeWarning: All-NaN slice encountered
  row_min = np.nanmin(mat, axis=1)
C:\Users\Talal Zahid\Desktop\Talal\Binance\features\ema_compression.py:66: RuntimeWarning: Mean of empty slice
  row_mean = np.nanmean(mat, axis=1)
C:\Users\Talal Zahid\Desktop\Talal\Binance\features\ema_compression.py:64: RuntimeWarning: All-N

In [11]:
# ============================================================
# BIG EMA RANGE SLOPE STRATEGY OPTIMIZER
# ~5,600 simulations
#
# Tests:
#   - different slope strengths / "angle" profiles
#   - different N candles for 1m / 5m / 15m
#   - slope-only strategy
#   - slope + 1m ribbon
#   - slope + all timeframe ribbons
#   - 15m compression breakout + 1m/5m slope
#   - 15m/5m slope + 1m trigger
# ============================================================

import time
import itertools
from pathlib import Path
from dataclasses import asdict, is_dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from simulation.rules import Rule, ALL
from simulation.simulator import Strategy, SimConfig, run_simulation


# ============================================================
# Main config
# ============================================================

INITIAL_CASH = 10_000
RESULTS_PATH = Path("optimization_results_ema_range_slope_big_5600.csv")
CHECKPOINT_EVERY = 25
MIN_TRADES_FOR_RANKING = 20

SIM_START = "2025-11-01"
SIM_END = "2026-05-25"

FEE_BPS = 10
SLIPPAGE_BPS = 1
MAX_OPEN_TRADES = 1


# ============================================================
# EMA compression/range columns
# These must already exist in merged from add_ema_compression_features()
# ============================================================

EMA_1M_RANGE = f"{EMA_1M_COMP_NAME}__range_pct"
EMA_5M_RANGE = f"{EMA_5M_COMP_NAME}__range_pct"
EMA_15M_RANGE = f"{EMA_15M_COMP_NAME}__range_pct"

EMA_15M_BULL_BREAKOUT = f"{EMA_15M_COMP_NAME}__bull_breakout"
EMA_5M_BULL_BREAKOUT = f"{EMA_5M_COMP_NAME}__bull_breakout"
EMA_1M_BULL_BREAKOUT = f"{EMA_1M_COMP_NAME}__bull_breakout"


required_cols = [
    "t", "open", "high", "low", "close",
    EMA_1M_RANGE,
    EMA_5M_RANGE,
    EMA_15M_RANGE,
    EMA50_1m,
    EMA100_1m,
    EMA150_1m,
    EMA50_5m,
    EMA100_5m,
    EMA150_5m,
    EMA50_15m,
    EMA100_15m,
    EMA150_15m,
]

optional_cols = [
    EMA_1M_BULL_BREAKOUT,
    EMA_5M_BULL_BREAKOUT,
    EMA_15M_BULL_BREAKOUT,
]

missing = [c for c in required_cols if c not in merged.columns]
if missing:
    raise KeyError(f"Missing required columns in merged: {missing}")

available_optional_cols = [c for c in optional_cols if c in merged.columns]
missing_optional_cols = [c for c in optional_cols if c not in merged.columns]

if missing_optional_cols:
    print("Optional breakout columns missing. Related strategy modes will still run as False unless available:")
    print(missing_optional_cols)


# ============================================================
# Parameter search space
# ============================================================

# Includes longer windows like 10 and 20.
N_1M_VALUES = (2, 3, 5, 10)
N_5M_VALUES = (1, 2, 5, 10)
N_15M_VALUES = (1, 2, 5, 10, 20)

# More slope "angle" variations.
# Values are percentage-points of EMA range %, not price percent.
#
# Example:
# range_pct goes from 0.100 to 0.120
# total_delta = 0.020
#
# step = minimum increase per timeframe step
# total = minimum increase across full N-window
SLOPE_STRENGTH_PROFILES = [
    {
        "name": "micro",
        "1m_step": 0.0005,
        "1m_total": 0.003,
        "5m_step": 0.0010,
        "5m_total": 0.005,
        "15m_step": 0.0010,
        "15m_total": 0.005,
    },
    {
        "name": "very_loose",
        "1m_step": 0.001,
        "1m_total": 0.005,
        "5m_step": 0.002,
        "5m_total": 0.008,
        "15m_step": 0.002,
        "15m_total": 0.008,
    },
    {
        "name": "loose",
        "1m_step": 0.002,
        "1m_total": 0.008,
        "5m_step": 0.003,
        "5m_total": 0.010,
        "15m_step": 0.003,
        "15m_total": 0.010,
    },
    {
        "name": "balanced",
        "1m_step": 0.003,
        "1m_total": 0.010,
        "5m_step": 0.005,
        "5m_total": 0.015,
        "15m_step": 0.005,
        "15m_total": 0.015,
    },
    {
        "name": "strong",
        "1m_step": 0.005,
        "1m_total": 0.015,
        "5m_step": 0.008,
        "5m_total": 0.020,
        "15m_step": 0.008,
        "15m_total": 0.020,
    },
    {
        "name": "aggressive",
        "1m_step": 0.008,
        "1m_total": 0.025,
        "5m_step": 0.010,
        "5m_total": 0.030,
        "15m_step": 0.010,
        "15m_total": 0.030,
    },
    {
        "name": "burst",
        "1m_step": 0.003,
        "1m_total": 0.030,
        "5m_step": 0.004,
        "5m_total": 0.040,
        "15m_step": 0.004,
        "15m_total": 0.040,
    },
]


STRATEGY_MODES = (
    # Pure range expansion across all 3 timeframes.
    "all_mtf_slope",

    # All slopes + 1m bullish ribbon.
    "all_mtf_slope_1m_ribbon",

    # All slopes + 1m, 5m, 15m bullish ribbons.
    "all_mtf_slope_all_ribbons",

    # Successful idea: 15m compression breakout, then 1m and 5m range expansion confirm.
    "15m_breakout_plus_1m5m_slope",

    # Higher timeframe slope context, 1m price timing trigger.
    "5m15m_slope_1m_price_trigger",
)


CLOSE_MODES = (
    "close_below_1m_ema50",
    "close_below_1m_ema100",
)


# Expected:
# 4 * 4 * 5 * 7 * 5 * 2 = 5,600
expected_runs = (
    len(N_1M_VALUES)
    * len(N_5M_VALUES)
    * len(N_15M_VALUES)
    * len(SLOPE_STRENGTH_PROFILES)
    * len(STRATEGY_MODES)
    * len(CLOSE_MODES)
)

print(f"Expected simulations: {expected_runs:,}")


# ============================================================
# Helpers
# ============================================================

def _to_dataframe(obj):
    if obj is None:
        return pd.DataFrame()

    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    if isinstance(obj, list):
        if len(obj) == 0:
            return pd.DataFrame()

        rows = []
        for x in obj:
            if isinstance(x, dict):
                rows.append(x)
            elif is_dataclass(x):
                rows.append(asdict(x))
            elif hasattr(x, "__dict__"):
                rows.append(vars(x))
            else:
                rows.append(x)

        return pd.DataFrame(rows)

    if isinstance(obj, dict):
        return pd.DataFrame([obj])

    return pd.DataFrame(obj)


def strong_slope_signal(
    values,
    tf_step: int,
    n: int,
    min_step_delta: float,
    min_total_delta: float,
) -> np.ndarray:
    """
    Vectorized EMA range slope strength check.

    tf_step:
      1  = 1m on 1m base chart
      5  = 5m on 1m base chart
      15 = 15m on 1m base chart

    n:
      Number of timeframe candles to check.

    min_step_delta:
      Minimum increase required at each timeframe step.

    min_total_delta:
      Minimum total increase over the full n-window.
    """
    arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
    length = len(arr)

    ok = np.ones(length, dtype=bool)

    min_required_shift = int(n) * int(tf_step)
    ok[:min_required_shift] = False

    prev_total = np.empty(length, dtype=float)
    prev_total[:] = np.nan
    prev_total[min_required_shift:] = arr[:-min_required_shift]

    total_delta = arr - prev_total

    ok &= np.isfinite(arr)
    ok &= np.isfinite(prev_total)
    ok &= total_delta >= float(min_total_delta)

    for k in range(int(n)):
        cur_shift = k * int(tf_step)
        prev_shift = (k + 1) * int(tf_step)

        cur = np.empty(length, dtype=float)
        prev = np.empty(length, dtype=float)

        cur[:] = np.nan
        prev[:] = np.nan

        if cur_shift == 0:
            cur[:] = arr
        else:
            cur[cur_shift:] = arr[:-cur_shift]

        prev[prev_shift:] = arr[:-prev_shift]

        step_delta = cur - prev

        ok &= np.isfinite(cur)
        ok &= np.isfinite(prev)
        ok &= step_delta >= float(min_step_delta)

    return ok


def ribbon_1m_bullish(df: pd.DataFrame) -> np.ndarray:
    return (
        (df[EMA50_1m].to_numpy(dtype=float) > df[EMA100_1m].to_numpy(dtype=float))
        & (df[EMA100_1m].to_numpy(dtype=float) > df[EMA150_1m].to_numpy(dtype=float))
    )


def ribbon_5m_bullish(df: pd.DataFrame) -> np.ndarray:
    return (
        (df[EMA50_5m].to_numpy(dtype=float) > df[EMA100_5m].to_numpy(dtype=float))
        & (df[EMA100_5m].to_numpy(dtype=float) > df[EMA150_5m].to_numpy(dtype=float))
    )


def ribbon_15m_bullish(df: pd.DataFrame) -> np.ndarray:
    return (
        (df[EMA50_15m].to_numpy(dtype=float) > df[EMA100_15m].to_numpy(dtype=float))
        & (df[EMA100_15m].to_numpy(dtype=float) > df[EMA150_15m].to_numpy(dtype=float))
    )


def close_signal(df: pd.DataFrame, mode: str) -> np.ndarray:
    close = df["close"].to_numpy(dtype=float)

    if mode == "close_below_1m_ema50":
        return close < df[EMA50_1m].to_numpy(dtype=float)

    if mode == "close_below_1m_ema100":
        return close < df[EMA100_1m].to_numpy(dtype=float)

    raise ValueError(f"Unknown close_mode: {mode}")


def make_sim_config():
    kwargs = dict(
        initial_cash=INITIAL_CASH,
        max_open_trades=MAX_OPEN_TRADES,
        fee_bps=FEE_BPS,
        slippage_bps=SLIPPAGE_BPS,
        sim_start=SIM_START,
        sim_end=SIM_END,
        sim_tz=tz,
    )

    try:
        return SimConfig(
            **kwargs,
            log_level="WARNING",
            progress=False,
            progress_bar=False,
        )
    except TypeError:
        return SimConfig(**kwargs)


def summarize_result(res, params: dict) -> dict:
    trades = _to_dataframe(getattr(res, "trades", None))

    if trades.empty or "pnl" not in trades.columns:
        closed = pd.DataFrame()
    else:
        closed = trades[trades["pnl"].notna()].copy()

    num_trades = len(closed)

    if num_trades:
        pnl = float(closed["pnl"].sum())
        wins = closed[closed["pnl"] > 0]
        losses = closed[closed["pnl"] < 0]

        win_rate = float(len(wins) / num_trades * 100)
        avg_pnl = float(closed["pnl"].mean())
        avg_winner = float(wins["pnl"].mean()) if len(wins) else np.nan
        avg_loser = float(losses["pnl"].mean()) if len(losses) else np.nan
        best_trade = float(closed["pnl"].max())
        worst_trade = float(closed["pnl"].min())
    else:
        pnl = 0.0
        win_rate = 0.0
        avg_pnl = 0.0
        avg_winner = np.nan
        avg_loser = np.nan
        best_trade = np.nan
        worst_trade = np.nan

    stats = getattr(res, "stats", {})
    if isinstance(stats, pd.Series):
        stats = stats.to_dict()

    final_equity = INITIAL_CASH + pnl
    profit_factor = np.nan
    max_dd = np.nan
    return_pct = (final_equity / INITIAL_CASH - 1) * 100

    if isinstance(stats, dict):
        final_equity = float(stats.get("final_cash", final_equity))
        profit_factor = float(stats.get("profit_factor", np.nan))
        max_dd = float(stats.get("max_drawdown_pct", np.nan))
        return_pct = float(stats.get("total_return_pct", return_pct))

    score = pnl

    if np.isfinite(max_dd):
        score -= max_dd * 25.0

    if np.isfinite(profit_factor):
        score += min(profit_factor, 5.0) * 25.0

    if num_trades < MIN_TRADES_FOR_RANKING:
        score -= 500.0

    return {
        **params,
        "num_trades": num_trades,
        "win_rate": win_rate,
        "pnl": pnl,
        "final_equity": final_equity,
        "return_pct": return_pct,
        "profit_factor": profit_factor,
        "max_drawdown_pct": max_dd,
        "avg_pnl": avg_pnl,
        "avg_winner": avg_winner,
        "avg_loser": avg_loser,
        "best_trade": best_trade,
        "worst_trade": worst_trade,
        "score": score,
    }


# ============================================================
# Prepare slim dataframe
# ============================================================

needed_cols = set(required_cols)
needed_cols.update(available_optional_cols)

sim_df = merged[list(needed_cols)].copy()
sim_df = sim_df.sort_values("t").reset_index(drop=True)
sim_df = sim_df.copy()

print("sim_df shape:", sim_df.shape)


# ============================================================
# Precompute reusable static signals
# ============================================================

ribbon_1m = ribbon_1m_bullish(sim_df)
ribbon_5m = ribbon_5m_bullish(sim_df)
ribbon_15m = ribbon_15m_bullish(sim_df)
ribbon_all = ribbon_1m & ribbon_5m & ribbon_15m

close_above_1m_ema50 = sim_df["close"].to_numpy(dtype=float) > sim_df[EMA50_1m].to_numpy(dtype=float)

if EMA_15M_BULL_BREAKOUT in sim_df.columns:
    breakout_15m = sim_df[EMA_15M_BULL_BREAKOUT].fillna(False).to_numpy(dtype=bool)
else:
    breakout_15m = np.zeros(len(sim_df), dtype=bool)


# ============================================================
# Signal caches
# ============================================================

slope_cache = {}
close_cache = {}


def get_slope_cached(tf: str, n: int, profile: dict) -> np.ndarray:
    key = (tf, int(n), profile["name"])

    if key in slope_cache:
        return slope_cache[key]

    if tf == "1m":
        sig = strong_slope_signal(
            sim_df[EMA_1M_RANGE],
            tf_step=1,
            n=n,
            min_step_delta=profile["1m_step"],
            min_total_delta=profile["1m_total"],
        )
    elif tf == "5m":
        sig = strong_slope_signal(
            sim_df[EMA_5M_RANGE],
            tf_step=5,
            n=n,
            min_step_delta=profile["5m_step"],
            min_total_delta=profile["5m_total"],
        )
    elif tf == "15m":
        sig = strong_slope_signal(
            sim_df[EMA_15M_RANGE],
            tf_step=15,
            n=n,
            min_step_delta=profile["15m_step"],
            min_total_delta=profile["15m_total"],
        )
    else:
        raise ValueError(f"Unknown tf: {tf}")

    slope_cache[key] = sig
    return sig


def get_close_cached(mode: str) -> np.ndarray:
    if mode not in close_cache:
        close_cache[mode] = close_signal(sim_df, mode)

    return close_cache[mode]


def build_open_signal(
    strategy_mode: str,
    slope_1m: np.ndarray,
    slope_5m: np.ndarray,
    slope_15m: np.ndarray,
) -> np.ndarray:
    if strategy_mode == "all_mtf_slope":
        return slope_1m & slope_5m & slope_15m

    if strategy_mode == "all_mtf_slope_1m_ribbon":
        return slope_1m & slope_5m & slope_15m & ribbon_1m

    if strategy_mode == "all_mtf_slope_all_ribbons":
        return slope_1m & slope_5m & slope_15m & ribbon_all

    if strategy_mode == "15m_breakout_plus_1m5m_slope":
        return breakout_15m & slope_1m & slope_5m & ribbon_15m

    if strategy_mode == "5m15m_slope_1m_price_trigger":
        return slope_5m & slope_15m & close_above_1m_ema50 & ribbon_1m

    raise ValueError(f"Unknown strategy_mode: {strategy_mode}")


# ============================================================
# Strategy object reads precomputed signal columns
# ============================================================

slope_strategy = Strategy(
    open_rules_long=ALL(
        Rule("EMA range slope optimizer open signal", lambda c: c.flag("__slope_open_signal"))
    ),
    close_rules_long=ALL(
        Rule("EMA range slope optimizer close signal", lambda c: c.flag("__slope_close_signal"))
    ),
)


# ============================================================
# Resume support
# ============================================================

if RESULTS_PATH.exists():
    previous = pd.read_csv(RESULTS_PATH)
    done_keys = set(previous["key"].astype(str))
    results = previous.to_dict("records")
    print(f"Resuming | completed={len(done_keys):,}")
else:
    done_keys = set()
    results = []


# ============================================================
# Grid
# ============================================================

grid = list(itertools.product(
    N_1M_VALUES,
    N_5M_VALUES,
    N_15M_VALUES,
    SLOPE_STRENGTH_PROFILES,
    STRATEGY_MODES,
    CLOSE_MODES,
))

print(f"Total simulations: {len(grid):,}")
print(f"Already completed: {len(done_keys):,}")
print(f"Remaining: {len(grid) - len(done_keys):,}")


# ============================================================
# Run optimizer
# ============================================================

t0 = time.time()
new_runs = 0

for (
    n_1m,
    n_5m,
    n_15m,
    profile,
    strategy_mode,
    close_mode,
) in tqdm(grid, desc="Optimizing EMA range slope big"):

    key = (
        f"strategy_{strategy_mode}__"
        f"n1m_{n_1m}_n5m_{n_5m}_n15m_{n_15m}__"
        f"profile_{profile['name']}__"
        f"close_{close_mode}"
    )

    if key in done_keys:
        continue

    slope_1m = get_slope_cached("1m", n_1m, profile)
    slope_5m = get_slope_cached("5m", n_5m, profile)
    slope_15m = get_slope_cached("15m", n_15m, profile)

    open_signal = build_open_signal(
        strategy_mode=strategy_mode,
        slope_1m=slope_1m,
        slope_5m=slope_5m,
        slope_15m=slope_15m,
    )

    exit_signal = get_close_cached(close_mode)

    sim_df["__slope_open_signal"] = open_signal
    sim_df["__slope_close_signal"] = exit_signal

    res = run_simulation(
        df=sim_df,
        strategy=slope_strategy,
        cfg=make_sim_config(),
        time_col="t",
        price_col="close",
    )

    params = {
        "key": key,

        "strategy_mode": strategy_mode,

        "n_1m": n_1m,
        "n_5m": n_5m,
        "n_15m": n_15m,

        "profile": profile["name"],

        "step_1m": profile["1m_step"],
        "total_1m": profile["1m_total"],

        "step_5m": profile["5m_step"],
        "total_5m": profile["5m_total"],

        "step_15m": profile["15m_step"],
        "total_15m": profile["15m_total"],

        "close_mode": close_mode,
    }

    results.append(summarize_result(res, params))
    done_keys.add(key)
    new_runs += 1

    if new_runs % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

        elapsed_min = (time.time() - t0) / 60
        avg_sec = (time.time() - t0) / max(new_runs, 1)
        remaining = len(grid) - len(done_keys)
        eta_h = avg_sec * remaining / 3600

        print(
            f"Checkpoint saved | new_runs={new_runs:,} | "
            f"total_done={len(done_keys):,} | elapsed={elapsed_min:.1f} min | "
            f"avg={avg_sec:.2f}s/run | ETA={eta_h:.2f}h"
        )


pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

print(f"Done. Saved: {RESULTS_PATH}")

Expected simulations: 5,600
sim_df shape: (295284, 20)
Total simulations: 5,600
Already completed: 0
Remaining: 5,600


Optimizing EMA range slope big:   0%|          | 0/5600 [00:00<?, ?it/s]

Checkpoint saved | new_runs=25 | total_done=25 | elapsed=1.9 min | avg=4.54s/run | ETA=7.04h
Checkpoint saved | new_runs=50 | total_done=50 | elapsed=3.7 min | avg=4.46s/run | ETA=6.88h
Checkpoint saved | new_runs=75 | total_done=75 | elapsed=5.5 min | avg=4.41s/run | ETA=6.77h
Checkpoint saved | new_runs=100 | total_done=100 | elapsed=7.3 min | avg=4.35s/run | ETA=6.65h
Checkpoint saved | new_runs=125 | total_done=125 | elapsed=9.0 min | avg=4.32s/run | ETA=6.57h
Checkpoint saved | new_runs=150 | total_done=150 | elapsed=10.7 min | avg=4.29s/run | ETA=6.50h
Checkpoint saved | new_runs=175 | total_done=175 | elapsed=12.5 min | avg=4.27s/run | ETA=6.44h
Checkpoint saved | new_runs=200 | total_done=200 | elapsed=14.2 min | avg=4.27s/run | ETA=6.41h
Checkpoint saved | new_runs=225 | total_done=225 | elapsed=15.9 min | avg=4.25s/run | ETA=6.34h
Checkpoint saved | new_runs=250 | total_done=250 | elapsed=17.7 min | avg=4.24s/run | ETA=6.30h
Checkpoint saved | new_runs=275 | total_done=275 | 

In [12]:
opt = pd.read_csv(RESULTS_PATH)

valid = opt[opt["num_trades"] >= MIN_TRADES_FOR_RANKING].copy()

best_by_profit = valid.sort_values(
    ["pnl", "final_equity", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_score = valid.sort_values(
    ["score", "pnl", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_winrate = valid.sort_values(
    ["win_rate", "pnl", "num_trades"],
    ascending=False,
)

best_by_profit_factor = valid.sort_values(
    ["profit_factor", "pnl", "num_trades"],
    ascending=False,
)

print("\nBEST BY PROFIT")
display(best_by_profit.head(30))

print("\nBEST BY RISK-ADJUSTED SCORE")
display(best_by_score.head(30))

print("\nBEST BY WIN RATE")
display(best_by_winrate.head(30))

print("\nBEST BY PROFIT FACTOR")
display(best_by_profit_factor.head(30))


print("\nSUMMARY BY STRATEGY MODE")
display(
    valid.groupby("strategy_mode")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
        avg_max_dd=("max_drawdown_pct", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)


print("\nSUMMARY BY PROFILE")
display(
    valid.groupby("profile")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
        avg_max_dd=("max_drawdown_pct", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)


print("\nSUMMARY BY N VALUES")
display(
    valid.groupby(["n_1m", "n_5m", "n_15m"])
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)


print("\nSUMMARY BY CLOSE MODE")
display(
    valid.groupby("close_mode")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)


BEST BY PROFIT


,key,strategy_mode,n_1m,n_5m,n_15m,profile,step_1m,total_1m,step_5m,total_5m,...,final_equity,return_pct,profit_factor,max_drawdown_pct,avg_pnl,avg_winner,avg_loser,best_trade,worst_trade,score
1814,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,2,1,burst,0.003,0.030,0.004,0.040,...,10002.289660,0.022897,1.002983,3.089287,0.095403,69.987832,-59.044345,190.048836,-125.619038,-49.867931
1815,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,2,1,burst,0.003,0.030,0.004,0.040,...,9997.746039,-0.022540,0.997009,4.086269,-0.107331,83.486106,-62.802409,245.873457,-176.633060,-79.485453
2164,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,5,1,burst,0.003,0.030,0.004,0.040,...,9971.857701,-0.281423,0.961688,3.784059,-1.340109,70.641292,-66.777747,190.048836,-125.619038,-98.701579
5285,strategy_all_mtf_slope_all_ribbons__n1m_10_n5m...,all_mtf_slope_all_ribbons,10,10,1,balanced,0.003,0.010,0.005,0.015,...,9916.374936,-0.836251,0.889018,5.233883,-3.345003,95.696633,-41.861194,284.007213,-116.616715,-192.246696
1813,strategy_all_mtf_slope_1m_ribbon__n1m_3_n5m_2_...,all_mtf_slope_1m_ribbon,3,2,1,burst,0.003,0.030,0.004,0.040,...,9900.042209,-0.999578,0.897366,4.509289,-3.844530,87.396514,-60.870183,245.873457,-176.633060,-190.255862
2495,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,10,1,strong,0.005,0.015,0.008,0.020,...,9847.903604,-1.520964,0.804926,3.996990,-7.242686,104.598341,-51.979096,202.635857,-133.176509,-231.897995
5284,strategy_all_mtf_slope_all_ribbons__n1m_10_n5m...,all_mtf_slope_all_ribbons,10,10,1,balanced,0.003,0.010,0.005,0.015,...,9842.336528,-1.576635,0.759632,3.324386,-5.839388,49.826217,-38.583861,145.807436,-79.679992,-221.782322
54,strategy_all_mtf_slope_all_ribbons__n1m_2_n5m_...,all_mtf_slope_all_ribbons,2,1,1,aggressive,0.008,0.025,0.010,0.030,...,9834.660383,-1.653396,0.757261,3.403768,-7.515437,51.580282,-56.761870,134.872838,-125.619038,-231.502284
5424,strategy_all_mtf_slope_all_ribbons__n1m_10_n5m...,all_mtf_slope_all_ribbons,10,10,5,balanced,0.003,0.010,0.005,0.015,...,9815.715207,-1.842848,0.659202,2.598002,-9.214240,71.292051,-36.049670,145.807436,-79.679992,-232.754797
198,strategy_5m15m_slope_1m_price_trigger__n1m_2_n...,5m15m_slope_1m_price_trigger,2,1,5,aggressive,0.008,0.025,0.010,0.030,...,9805.764107,-1.942359,0.719149,2.884184,-7.769436,82.893490,-36.399833,172.560633,-106.794391,-248.361780



BEST BY RISK-ADJUSTED SCORE


,key,strategy_mode,n_1m,n_5m,n_15m,profile,step_1m,total_1m,step_5m,total_5m,...,final_equity,return_pct,profit_factor,max_drawdown_pct,avg_pnl,avg_winner,avg_loser,best_trade,worst_trade,score
1814,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,2,1,burst,0.003,0.030,0.004,0.040,...,10002.289660,0.022897,1.002983,3.089287,0.095403,69.987832,-59.044345,190.048836,-125.619038,-49.867931
1815,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,2,1,burst,0.003,0.030,0.004,0.040,...,9997.746039,-0.022540,0.997009,4.086269,-0.107331,83.486106,-62.802409,245.873457,-176.633060,-79.485453
2164,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,5,1,burst,0.003,0.030,0.004,0.040,...,9971.857701,-0.281423,0.961688,3.784059,-1.340109,70.641292,-66.777747,190.048836,-125.619038,-98.701579
1813,strategy_all_mtf_slope_1m_ribbon__n1m_3_n5m_2_...,all_mtf_slope_1m_ribbon,3,2,1,burst,0.003,0.030,0.004,0.040,...,9900.042209,-0.999578,0.897366,4.509289,-3.844530,87.396514,-60.870183,245.873457,-176.633060,-190.255862
5285,strategy_all_mtf_slope_all_ribbons__n1m_10_n5m...,all_mtf_slope_all_ribbons,10,10,1,balanced,0.003,0.010,0.005,0.015,...,9916.374936,-0.836251,0.889018,5.233883,-3.345003,95.696633,-41.861194,284.007213,-116.616715,-192.246696
5284,strategy_all_mtf_slope_all_ribbons__n1m_10_n5m...,all_mtf_slope_all_ribbons,10,10,1,balanced,0.003,0.010,0.005,0.015,...,9842.336528,-1.576635,0.759632,3.324386,-5.839388,49.826217,-38.583861,145.807436,-79.679992,-221.782322
54,strategy_all_mtf_slope_all_ribbons__n1m_2_n5m_...,all_mtf_slope_all_ribbons,2,1,1,aggressive,0.008,0.025,0.010,0.030,...,9834.660383,-1.653396,0.757261,3.403768,-7.515437,51.580282,-56.761870,134.872838,-125.619038,-231.502284
2495,strategy_all_mtf_slope_all_ribbons__n1m_3_n5m_...,all_mtf_slope_all_ribbons,3,10,1,strong,0.005,0.015,0.008,0.020,...,9847.903604,-1.520964,0.804926,3.996990,-7.242686,104.598341,-51.979096,202.635857,-133.176509,-231.897995
5424,strategy_all_mtf_slope_all_ribbons__n1m_10_n5m...,all_mtf_slope_all_ribbons,10,10,5,balanced,0.003,0.010,0.005,0.015,...,9815.715207,-1.842848,0.659202,2.598002,-9.214240,71.292051,-36.049670,145.807436,-79.679992,-232.754797
198,strategy_5m15m_slope_1m_price_trigger__n1m_2_n...,5m15m_slope_1m_price_trigger,2,1,5,aggressive,0.008,0.025,0.010,0.030,...,9805.764107,-1.942359,0.719149,2.884184,-7.769436,82.893490,-36.399833,172.560633,-106.794391,-248.361780



BEST BY WIN RATE


...

,tests,avg_pnl,median_pnl,best_pnl,avg_win_rate,avg_profit_factor,avg_max_dd
profile,,,,,,,
burst,366,-3943.581530,-2586.825548,2.289660,17.804590,0.293878,40.004112
balanced,542,-4603.336615,-3292.969114,-83.625064,14.397525,0.237045,46.176913
strong,449,-3917.626255,-2227.561448,-152.096396,17.579550,0.279805,39.526675
aggressive,350,-3042.209388,-1548.415777,-165.339617,20.600551,0.297733,31.021012
loose,616,-5259.931344,-4826.807898,-309.245495,14.393481,0.215450,52.725286
very_loose,626,-5979.300294,-6107.756009,-502.171853,12.905470,0.187227,59.865719
micro,638,-6741.087034,-7775.990478,-621.520809,12.120795,0.173434,67.477638



SUMMARY BY N VALUES


tests      avg_pnl   median_pnl     best_pnl  avg_win_rate  \
n_1m n_5m n_15m                                                               
3    2    1         56 -6368.874785 -7199.886783     2.289660     17.046599   
     5    1         55 -5923.489962 -6125.939429   -28.142299     16.156803   
10   10   1         45 -4859.688178 -3535.486786   -83.625064     14.805869   
3    10   1         48 -5245.989140 -4904.997008  -152.096396     13.919011   
2    1    1         50 -6084.837851 -6198.419636  -165.339617     17.554099   
...                ...          ...          ...          ...           ...   
3    10   5         44 -5284.129817 -4035.375089  -894.837219     12.010097   
10   2    2         56 -6249.209204 -6604.511314  -903.977449     14.483273   
2    10   10        39 -4076.308222 -2731.793991  -996.872047     11.850785   
     2    5         48 -5756.045256 -5817.037886 -1016.923077     14.581965   
5    2    2         56 -6509.002621 -7481.194571 -1035.748488     14.176348   

                 avg_profit_factor  
n_1m n_5m n_15m                     
3    2    1               0.273757  
     5    1               0.266532  
10   10   1               0.266856  
3    10   1               0.263172  
2    1    1               0.257174  
...                            ...  
3    10   5               0.210006  
10   2    2               0.204324  
2    10   10              0.157970  
     2    5               0.231896  
5    2    2               0.217251  

[80 rows x 6 columns]


SUMMARY BY CLOSE MODE


,tests,avg_pnl,median_pnl,best_pnl,avg_win_rate,avg_profit_factor
close_mode,,,,,,
close_below_1m_ema50,1824,-5281.655882,-4533.305129,2.289660,14.730178,0.213039
close_below_1m_ema100,1763,-4771.627709,-3758.179468,-2.253961,15.447408,0.248363


In [ ]:


####################################################################################
N_CONFIRM = 1               # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 100
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 10 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

N_SPREAD_CONFIRM = 3
MIN_SPREAD_PCT = 0.1

MIN_EMAS_CROSSED = 4
CROSS_LOOKBACK = 40


#####################################################################################


LONG_EMA_FILTERS = [
    EMA20_1m
    # EMA50_1m,
    # EMA100_1m,
    # EMA150_1m,
    # EMA200_1m,
    # EMA100_5m,
    # EMA200_5m,
    # EMA100_15m,
    #EMA200_15m
]


macd_rules = ALL(
    Rule(
        "1m MACD above threshold",
        lambda c: c.gt(MACD_1M, 10)
    ),
    
    # Rule(
    #     "5m MACD above threshold",
    #     lambda c: c.gt(MACD_5M, MACD_THRESHOLD_5M)
    # ),
    
    # Rule(
    #     "15m MACD above threshold",
    #     lambda c: c.gt(MACD_15M, MACD_THRESHOLD_15M)
    # ),

    Rule(
        "1m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_1M, 50)
    ),
    
    Rule(
         "5m MACD - Signal above minimum",
         lambda c: c.gt(MACD_HIST_5M, 20)
     ),

    Rule(
        "15m MACD - Signal above minimum",
        lambda c: c.gt(MACD_HIST_15M, 20)
    ),
    
)


open_rules_long = ALL(
    macd_rules,
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_below_all(LONG_EMA_FILTERS)

),
    Rule(
        f"Last {N_CONFIRM} candles green",
        lambda c: c.consecutive_green(n=N_CONFIRM)
    ),
)



close_rules_long = ALL(
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_below_all(LONG_EMA_FILTERS)

),
)




###################### stochastic  rules #########################


X_1M = 5
Y_5M = 6
Z_15M = 3

open_rules_long_stoch_recovery = ALL(

    Rule(
        "1m MACD above threshold",
        lambda c: c.gt(MACD_1M, 100)
    ),
    
    # Rule(
    #     "5m MACD above threshold",
    #     lambda c: c.gt(MACD_5M, 5)
    # ),
    
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all(LONG_EMA_FILTERS)

    ),
    # Rule(
    #     "1m MACD - Signal above minimum",
    #     lambda c: c.lt(EMA50_5m, EMA100_5m)
    # ),
    # Rule(
    #     f"1m stochastic K crossed up 20 in last {1} 1m candles and stayed above",
    #     lambda c: c.flag(f"1m__st14__K_CROSS_UP_20_ACTIVE_{1}")
    # ),

    Rule(
        f"5m stochastic K crossed up 20 in last {Y_5M} 5m candles and stayed above",
        lambda c: c.flag(f"5m__st14__K_CROSS_UP_20_ACTIVE_{Y_5M}")
    ),

    Rule(
        "15m Stochastic K > D",
        lambda c: c.gt(STO_K_15M, STO_D_15M)
    ),
    Rule(
        "5m Stochastic K > D",
        lambda c: c.gt(STO_K_5M, STO_D_5M)
    ),
    # Rule(
    #     f"15m stochastic K crossed up 20 in last {Z_15M} 15m candles and stayed above",
    #     lambda c: c.flag(f"15m__st14__K_CROSS_UP_20_ACTIVE_{Z_15M}")
    # ),
    Rule(
        f"1m stochastic K crossed up 20 in last {1} 1m candles and stayed above",
        lambda c: c.flag(f"1h__st14__K_CROSS_UP_20_ACTIVE_{1}")
    )
)

close_rules_long_stoch_recovery = ANY(
    Rule(
        "1m stochastic K crossed back below 20",
        lambda c: c.flag("1m__st14__K_CROSS_DOWN_20")
    ),

    Rule(
        "1m stochastic K crossed down from overbought 80",
        lambda c: c.flag("1m__st14__K_CROSS_DOWN_80")
    ),
)




In [ ]:
# ============================================================
# EMA compression feature names
# ============================================================

EMA_1M_COMP_NAME = "ema_1m_50_100_150_comp"
EMA_5M_COMP_NAME = "ema_5m_50_100_150_comp"
EMA_15M_COMP_NAME = "ema_15m_50_100_150_comp"
EMA_1H_COMP_NAME = "ema_1h_50_100_150_comp"

EMA_1M_RANGE = f"{EMA_1M_COMP_NAME}__range_pct"
EMA_5M_RANGE = f"{EMA_5M_COMP_NAME}__range_pct"
EMA_15M_RANGE = f"{EMA_15M_COMP_NAME}__range_pct"

N_1M_SLOPE = 3
N_5M_SLOPE = 2
N_15M_SLOPE = 2

# ============================================================
# Add EMA compression features for 5m, 15m, 1h
# ============================================================

merged = add_ema_compression_features(
    merged_tmp,
    specs=[
        EMACompressionSpec(
            name=EMA_1M_COMP_NAME,
            cols=[EMA50_1m, EMA100_1m, EMA150_1m],
            compress_thr=0.35,
            expand_thr=0.40,
            lookback=20,
            require_bullish_order=False,
            fresh_only=False,
        ),
        EMACompressionSpec(
            name=EMA_5M_COMP_NAME,
            cols=[EMA50_5m, EMA100_5m, EMA150_5m],
            compress_thr=0.15, #0.15
            expand_thr=0.3, #0.3
            lookback=20, #20
            require_bullish_order=True,
            fresh_only=False,
        ),

        EMACompressionSpec(
            name=EMA_15M_COMP_NAME,
            cols=[EMA50_15m, EMA100_15m, EMA150_15m],
            compress_thr=0.10,
            expand_thr=0.15,
            lookback=20,
            require_bullish_order=True,
            fresh_only=False,
        ),

        EMACompressionSpec(
            name=EMA_1H_COMP_NAME,
            cols=[EMA50_1h, EMA100_1h, EMA150_1h],
            compress_thr=0.10,
            expand_thr=0.15,
            lookback=20,
            require_bullish_order=True,
            fresh_only=False,
        ),
    ],
)

######## ema rules #####
LONG_EMA_FILTERS = [
    EMA20_1m,
    EMA50_15m
]

COMPRESSION_MAX = 0.7
COMPRESSION_MIN = 0.35

open_rules_long = ALL(
    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all([EMA100_5m])

    ),
    Rule(
        "1m MACD - Signal above minimum",
        lambda c: c.gt(EMA100_5m, EMA200_15m)
    ),
    Rule(
        "15m EMA50/100/150 compressed then expanded bullish",
        lambda c: c.flag(f"{EMA_1M_COMP_NAME}__bull_breakout")
    ),
    Rule(
        f"15m EMA compression between {COMPRESSION_MIN}% and {COMPRESSION_MAX}%",
        lambda c: (
            c.gte(f"{EMA_5M_COMP_NAME}__range_pct", COMPRESSION_MIN)
            and c.lte(f"{EMA_5M_COMP_NAME}__range_pct", COMPRESSION_MAX)
        )
    ),
)

# open_rules_long = ALL(
#     Rule(
#         "15m EMA50/100/150 compressed then expanded bullish",
#         lambda c: c.flag(f"{EMA_5M_COMP_NAME}__bull_breakout")
#     ),
# )


### slope rules
open_slope_rules_long = ALL(
    Rule(
        "1m EMA range % strongly rising",
        lambda c: range_slope_strong_for_n_tf_candles(
            c, EMA_1M_RANGE, tf_step=1, n=3,
            min_step_delta=0.003,
            min_total_delta=0.010,
        )
    ),

    Rule(
        "5m EMA range % strongly rising",
        lambda c: range_slope_strong_for_n_tf_candles(
            c, EMA_5M_RANGE, tf_step=5, n=2,
            min_step_delta=0.005,
            min_total_delta=0.015,
        )
    ),

    Rule(
        "15m EMA range % strongly rising",
        lambda c: range_slope_strong_for_n_tf_candles(
            c, EMA_15M_RANGE, tf_step=15, n=2,
            min_step_delta=0.005,
            min_total_delta=0.015,
        )
    ),


    Rule(
        "1m EMA ribbon bullish",
        lambda c: c.gt(EMA50_1m, EMA100_1m) and c.gt(EMA100_1m, EMA150_1m)
    ),

    Rule(
        "5m EMA ribbon bullish",
        lambda c: c.gt(EMA50_5m, EMA100_5m) and c.gt(EMA100_5m, EMA150_5m)
    ),

    Rule(
        "15m EMA ribbon bullish",
        lambda c: c.gt(EMA50_15m, EMA100_15m) and c.gt(EMA100_15m, EMA150_15m)
    ),
)

close_rules_long = ALL(
    Rule(
        "Close below 1m EMA50",
        lambda c: c.lt("close", EMA50_5m)
    ),
)
##################################################

strategy = Strategy(
    open_rules_long=open_slope_rules_long,
    close_rules_long=close_rules_long,
)

cfg = SimConfig(
    initial_cash=10_000,
    max_open_trades=1,
    fee_bps=10,
    slippage_bps=1,

    sim_start="2025-11-01",
    sim_end="2026-05-25",
    sim_tz=tz,

    exit=TradeExitConfig(
        enabled=True,
        # Important:
        # if both TP and SL are touched in same candle, assume SL first
        intrabar_priority="stop_first",
        # Important:
        # disables ST2 close rule while TP/SL system is active
        allow_rule_close=False,

        stop_loss=StopLossConfig(
            mode="entry_pct",
            value=1.25,   # 0.5% below entry for long
        ),
        sizing=PositionSizingConfig(
            mode="cash",  # use normal full cash sizing
        ),
        take_profits=(
            TakeProfitConfig(
                label="TP1",
                mode="entry_pct",
                value=1.5,          # +0.5%
                close_pct=100.0,     # close 50% of remaining
                # move_stop_mode="breakeven",
            ),

        ),
    ),
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=cfg,
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
ema_compression_specs = {
    "1m": IndicatorSpec("ema_compression", tag=EMA_1M_COMP_NAME, config={
        "feature_name": EMA_1M_COMP_NAME,
        "title": "1m EMA50/100/150 Compression (%)",
    }),

    "5m": IndicatorSpec("ema_compression", tag=EMA_5M_COMP_NAME, config={
        "feature_name": EMA_5M_COMP_NAME,
        "title": "5m EMA50/100/150 Compression (%)",
    }),

    "15m": IndicatorSpec("ema_compression", tag=EMA_15M_COMP_NAME, config={
        "feature_name": EMA_15M_COMP_NAME,
        "title": "15m EMA50/100/150 Compression (%)",
    }),

    "1h": IndicatorSpec("ema_compression", tag=EMA_1H_COMP_NAME, config={
        "feature_name": EMA_1H_COMP_NAME,
        "title": "1h EMA50/100/150 Compression (%)",
    }),
}

In [ ]:
indicators_by_tf = {
    "1m": [*ind_1m, ema_compression_specs["1m"]],
    "5m": [*ind_5m, ema_compression_specs["5m"]],
    "15m": [*ind_15m, ema_compression_specs["15m"]],
    "1h": [*ind_1h, ema_compression_specs["1h"]],
}

plot_toggles = [
    # Optional EMA overlays
    PlotToggle("1m", "ema", visible="legendonly"),
    PlotToggle("5m", "ema", visible="legendonly"),
    PlotToggle("15m", "ema", visible="legendonly"),
    PlotToggle("1h", "ema", visible="legendonly"),

    # Stochastic panels
    PlotToggle("1m", "st14", visible=False),
    PlotToggle("5m", "st14", visible=False),
    PlotToggle("15m", "st14", visible=False),
    PlotToggle("1h", "st14", visible=False),

    # Hide MACD
    PlotToggle("1m", "macd_8_21_5",  visible=False),
    PlotToggle("5m", "macd_8_21_5",  visible=False),
    PlotToggle("15m", "macd_8_21_5",  visible=False),
    PlotToggle("1h", "macd_8_21_5",  visible=False),

    PlotToggle("1m", EMA_1M_COMP_NAME, visible=True),
    PlotToggle("5m", EMA_5M_COMP_NAME, visible=True),
    PlotToggle("15m", EMA_15M_COMP_NAME, visible=True),
    PlotToggle("1h", EMA_1H_COMP_NAME, visible=True),

]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)

assert_plot_columns_exist(merged, plot_indicators)


In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        # date_from="2026-03-23",
        # date_to="2026-03-24",
        date_from="2025-11-9",
        date_to="2025-11-12",
        height=1400,
        width=1900,
    ),
)
